# 🌱 Module 3: Soil Analysis System
## Bharat Krishi AI — Intelligent Agriculture Decision Support System

**Dataset:** Soil_Nutrients.csv  
**Records:** 15,400 | **Features:** 19 | **Crops:** 22 | **Soil Types:** 3

| Feature | Description |
|---|---|
| Name | Crop name |
| Fertility | Target — High / Moderate |
| Nitrogen / Phosphorus / Potassium | Soil nutrients |
| pH / Category_pH | Soil acidity |
| Temperature / Rainfall | Climate conditions |
| Soil_Type | Loam / Sandy Loam / Sandy |
| Season | Summer / Spring / Fall / Winter |
| N_Ratio / P_Ratio / K_Ratio | Nutrient ratios |

---
## 📦 Section 1 — Install & Import Libraries

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn plotly --quiet

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
from xgboost import XGBClassifier

import joblib
sns.set_theme(style='whitegrid', palette='viridis')
print('✅ All libraries imported successfully!')

---
## 📂 Section 2 — Data Collection Questions (Q1–Q10)

In [ ]:
# Q1 — Load and inspect
df = pd.read_csv('Soil_Nutrients.csv')
print('Q1 — Dataset Overview:')
print(df.info())
print()
df.head(10)

In [ ]:
# Q2 — Total samples
print(f'Q2 — Total soil samples : {df.shape[0]:,}')
print(f'     Total features     : {df.shape[1]}')

In [ ]:
# Q3 — Nutrients measured
print('Q3 — Soil nutrients measured in the dataset:')
nutrients = {
    'Nitrogen'   : 'Macro-nutrient — essential for leaf and stem growth',
    'Phosphorus' : 'Macro-nutrient — essential for root and flower development',
    'Potassium'  : 'Macro-nutrient — essential for fruit quality and disease resistance',
    'pH'         : 'Soil acidity/alkalinity — affects nutrient availability',
    'N_Ratio'    : 'Nitrogen ratio relative to other nutrients',
    'P_Ratio'    : 'Phosphorus ratio relative to other nutrients',
    'K_Ratio'    : 'Potassium ratio relative to other nutrients'
}
for k, v in nutrients.items():
    print(f'  {k:15s}: {v}')

In [ ]:
# Q4 — Soil types
print('Q4 — Soil types in dataset:')
print(df['Soil_Type'].value_counts())

# Q5 — Regions / Crops
print(f'\nQ5 — Unique crops (Name): {df["Name"].nunique()}')
print(f'     Crops: {sorted(df["Name"].unique())}')

In [ ]:
# Q6 — NPK ranges
print('Q6 — NPK value ranges:')
for col in ['Nitrogen', 'Phosphorus', 'Potassium']:
    print(f'  {col:12s}: Min={df[col].min():.2f}  Max={df[col].max():.2f}  Mean={df[col].mean():.2f}  Std={df[col].std():.2f}')

In [ ]:
# Q7 — Numerical vs Categorical
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Q7 — Numerical  features ({len(num_cols)}): {num_cols}')
print(f'     Categorical features ({len(cat_cols)}): {cat_cols}')

In [ ]:
# Q8 — Fertility categories
print('Q8 — Soil fertility categories:')
print(df['Fertility'].value_counts())
print(f'     Total classes: {df["Fertility"].nunique()}')

# Q9 & Q10 — Useful and irrelevant columns
print('\nQ9  — Useful columns for fertility prediction:')
print('  Nitrogen, Phosphorus, Potassium, pH, Temperature, Rainfall,')
print('  Soil_Type, Season, N_Ratio, P_Ratio, K_Ratio → Features')
print('  Fertility → Target')
print('\nQ10 — Columns that may be dropped or used carefully:')
print('  Name (crop name) — not a soil property; can be used for group analysis')
print('  Category_pH — derived from pH, may cause redundancy')

---
## 🧹 Section 3 — Data Preprocessing Questions (Q1–Q12)

In [ ]:
# Q1 & Q2 — Missing values
missing = df.isnull().sum()
print('Q1 & Q2 — Missing Values:')
print(missing[missing > 0] if missing.sum() > 0 else '✅ No missing values!')
print(f'Total missing: {missing.sum()}')

In [ ]:
# Q3 — Imputation if needed
for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)
print('Q3 — Imputation applied (median/mode). Remaining missing:', df.isnull().sum().sum())

In [ ]:
# Q4, Q5, Q6 — Duplicates
dupes = df.duplicated().sum()
print(f'Q4 — Duplicates present : {dupes > 0}')
print(f'Q5 — Duplicate count    : {dupes}')
if dupes > 0:
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
print(f'Q6 — Dataset shape after check: {df.shape}')

In [ ]:
# Q7 — Consistent format check
print('Q7 — Checking nutrient value formats:')
for col in ['Nitrogen','Phosphorus','Potassium','pH']:
    print(f'  {col:12s}: dtype={df[col].dtype}  min={df[col].min():.3f}  max={df[col].max():.3f}')
print('  ✅ All nutrient columns are float — consistent format.')

In [ ]:
# Q8 — Encode categorical variables
df['Soil_Type']    = df['Soil_Type'].str.strip()
df['Season']       = df['Season'].str.strip()
df['Category_pH']  = df['Category_pH'].str.strip()
df['Photoperiod']  = df['Photoperiod'].str.strip()
df['Name']         = df['Name'].str.strip()

le_soil     = LabelEncoder()
le_season   = LabelEncoder()
le_ph_cat   = LabelEncoder()
le_photo    = LabelEncoder()
le_name     = LabelEncoder()
le_fertility= LabelEncoder()

df['Soil_Enc']     = le_soil.fit_transform(df['Soil_Type'])
df['Season_Enc']   = le_season.fit_transform(df['Season'])
df['pH_Cat_Enc']   = le_ph_cat.fit_transform(df['Category_pH'])
df['Photo_Enc']    = le_photo.fit_transform(df['Photoperiod'])
df['Name_Enc']     = le_name.fit_transform(df['Name'])
df['Fertility_Enc']= le_fertility.fit_transform(df['Fertility'])

print('Q8 — Encoding complete:')
print(f'  Soil types  : {dict(zip(le_soil.classes_, le_soil.transform(le_soil.classes_)))}')
print(f'  Seasons     : {dict(zip(le_season.classes_, le_season.transform(le_season.classes_)))}')
print(f'  Fertility   : {dict(zip(le_fertility.classes_, le_fertility.transform(le_fertility.classes_)))}')

In [ ]:
# Q9 — Scaling
print('Q9 — StandardScaler will be applied to numerical features before neural network / SVM training.')

# Q10 — Outlier detection (IQR)
print('\nQ10 — Outlier detection (IQR method):')
for col in ['Nitrogen','Phosphorus','Potassium','pH']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f'  {col:12s}: {outliers} outliers detected')

In [ ]:
# Q11 — Consistency
print('Q11 — Soil type consistency check:')
print(df['Soil_Type'].value_counts())
print('  ✅ All soil types are clean and consistent.')

# Q12 — Summary
print('\nQ12 — Cleaning Summary:')
print(f'  ✅ Missing values handled')
print(f'  ✅ Duplicates checked')
print(f'  ✅ Categorical variables encoded')
print(f'  ✅ Outliers identified')
print(f'  ✅ Final shape: {df.shape}')

---
## 🔍 Section 4 — EDA Questions (Q1–Q17)

In [ ]:
# Q1 — Most frequent soil type
print(f'Q1  — Most frequent soil type: {df["Soil_Type"].value_counts().idxmax()} ({df["Soil_Type"].value_counts().max()} records)')

# Q2 & Q3 — Highest and lowest nutrient concentration
means = df[['Nitrogen','Phosphorus','Potassium']].mean()
print(f'Q2  — Highest avg nutrient: {means.idxmax()} ({means.max():.2f})')
print(f'Q3  — Lowest  avg nutrient: {means.idxmin()} ({means.min():.2f})')

In [ ]:
# Q4 & Q5 — Soil type fertility
fertility_rate = df.groupby('Soil_Type')['Fertility'].apply(
    lambda x: (x == 'High').sum() / len(x) * 100
).sort_values(ascending=False)
print('Q4 & Q5 — Fertility rate (% High) by Soil Type:')
print(fertility_rate.round(2))
print(f'\nQ4 — Highest fertility soil: {fertility_rate.idxmax()}')
print(f'Q5 — Lowest  fertility soil: {fertility_rate.idxmin()}')

In [ ]:
# Q6, Q7, Q8 — NPK by soil type
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, nutrient, color in zip(axes, ['Nitrogen','Phosphorus','Potassium'], ['steelblue','seagreen','tomato']):
    df.boxplot(column=nutrient, by='Soil_Type', ax=ax, patch_artist=True)
    ax.set_title(f'{nutrient} by Soil Type', fontweight='bold')
    ax.set_xlabel('Soil Type')
    ax.set_ylabel(nutrient)
plt.suptitle('Q6/Q7/Q8 — NPK Distribution by Soil Type', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Q9 — pH vs Fertility
ph_fert = df.groupby('Fertility')['pH'].mean()
print('Q9 — Average pH by Fertility class:')
print(ph_fert.round(3))
corr_ph = df['pH'].corr(df['Fertility_Enc'])
print(f'     Correlation (pH vs Fertility_Enc): {corr_ph:.4f}')

plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='Fertility', y='pH', palette='Set2')
plt.title('Q9 — pH Distribution by Fertility Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Q10 — Nutrient deficiency detection
N_thresh = df['Nitrogen'].quantile(0.25)
P_thresh = df['Phosphorus'].quantile(0.25)
K_thresh = df['Potassium'].quantile(0.25)

n_def = (df['Nitrogen']   < N_thresh).sum()
p_def = (df['Phosphorus'] < P_thresh).sum()
k_def = (df['Potassium']  < K_thresh).sum()

print('Q10 — Nutrient deficiency (below 25th percentile):')
print(f'  Nitrogen   deficient: {n_def} ({n_def/len(df)*100:.1f}%)')
print(f'  Phosphorus deficient: {p_def} ({p_def/len(df)*100:.1f}%)')
print(f'  Potassium  deficient: {k_def} ({k_def/len(df)*100:.1f}%)')

In [ ]:
# Q11 & Q12 — Nutrient-rich vs poor crops
crop_fertility = df.groupby('Name')['Fertility'].apply(
    lambda x: (x=='High').sum()/len(x)*100
).sort_values(ascending=False)
print('Q11 — Crops with highest soil fertility rate (nutrient-rich):')
print(crop_fertility.head(5).round(1))
print('\nQ12 — Crops with lowest soil fertility rate (poor soil):')
print(crop_fertility.tail(5).round(1))

In [ ]:
# Q13 — Nutrient correlations
num_feats = ['Nitrogen','Phosphorus','Potassium','pH','Temperature','Rainfall',
             'Light_Hours','Light_Intensity','Rh','Yield','N_Ratio','P_Ratio','K_Ratio']
corr = df[num_feats].corr()
plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Q13 — Soil Nutrient Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Q14 — Nutrient contribution to fertility
print('Q14 — Mean nutrient values by Fertility class:')
print(df.groupby('Fertility')[['Nitrogen','Phosphorus','Potassium','pH']].mean().round(3))

In [ ]:
# Q15 — Seasonal patterns in soil quality
print('Q15 — Soil quality by Season:')
season_fert = df.groupby('Season')[['Nitrogen','Phosphorus','Potassium','pH']].mean().round(2)
print(season_fert)

# Q16 — Best soil types for agriculture
print('\nQ16 — Soil suitability for agriculture (avg yield by soil type):')
print(df.groupby('Soil_Type')['Yield'].mean().sort_values(ascending=False).round(3))

# Q17 — Fertility hotspot crops
print('\nQ17 — Soil fertility hotspot crops (High fertility %):')
print(crop_fertility.head(5).round(1))

---
## 📊 Section 5 — Visualization Questions (Q1–Q10)

In [ ]:
# Q1 — Distribution of soil nutrients
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
plot_cols = ['Nitrogen','Phosphorus','Potassium','pH','Temperature','Rainfall','N_Ratio','Yield']
colors = sns.color_palette('viridis', len(plot_cols))
for i, col in enumerate(plot_cols):
    axes[i].hist(df[col], bins=40, color=colors[i], edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Value')
plt.suptitle('Q1 — Soil Nutrient & Feature Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Q2 — Fertility distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fertility_counts = df['Fertility'].value_counts()
axes[0].bar(fertility_counts.index, fertility_counts.values,
            color=['#2ecc71','#f39c12'], edgecolor='black', width=0.5)
for i, v in enumerate(fertility_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')
axes[0].set_title('Fertility Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
axes[1].pie(fertility_counts.values, labels=fertility_counts.index,
            autopct='%1.1f%%', colors=['#2ecc71','#f39c12'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Fertility Class Proportion', fontweight='bold')
plt.suptitle('Q2 — Soil Fertility Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Q3 — Nutrient levels by crop
crop_npk = df.groupby('Name')[['Nitrogen','Phosphorus','Potassium']].mean().sort_values('Nitrogen', ascending=False)
crop_npk.plot(kind='bar', figsize=(16, 6), colormap='Set2', edgecolor='black', width=0.75)
plt.title('Q3 — Average NPK by Crop', fontsize=14, fontweight='bold')
plt.xlabel('Crop')
plt.ylabel('Average Nutrient Value')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Nutrient')
plt.tight_layout()
plt.show()

In [ ]:
# Q5 — pH distribution across soil types
plt.figure(figsize=(10, 5))
sns.violinplot(data=df, x='Soil_Type', y='pH', hue='Fertility', split=True, palette='Set2')
plt.title('Q5 — pH Distribution by Soil Type and Fertility', fontsize=13, fontweight='bold')
plt.xlabel('Soil Type')
plt.ylabel('pH')
plt.tight_layout()
plt.show()

In [ ]:
# Q9 — Scatter: pH vs Nitrogen, colored by Fertility
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, nutrient in zip(axes, ['Nitrogen','Phosphorus','Potassium']):
    for fert, color in zip(['High','Moderate'], ['#2ecc71','#f39c12']):
        sub = df[df['Fertility'] == fert]
        ax.scatter(sub['pH'], sub[nutrient], alpha=0.3, s=8, color=color, label=fert)
    ax.set_title(f'Q9 — pH vs {nutrient}', fontweight='bold')
    ax.set_xlabel('pH')
    ax.set_ylabel(nutrient)
    ax.legend(title='Fertility', fontsize=8)
plt.suptitle('pH vs Soil Nutrients by Fertility', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔧 Section 6 — Feature Engineering Questions (Q1–Q8)

In [ ]:
# Q1 — Soil fertility score from nutrients
df['Total_NPK'] = df['Nitrogen'] + df['Phosphorus'] + df['Potassium']
df['NPK_Mean']  = df['Total_NPK'] / 3

# Q2 — NPK combined already in Total_NPK
# Q3 — Nutrient ratios
df['N_P_Ratio'] = df['Nitrogen'] / (df['Phosphorus'] + 1e-5)
df['N_K_Ratio'] = df['Nitrogen'] / (df['Potassium']  + 1e-5)
df['P_K_Ratio'] = df['Phosphorus'] / (df['Potassium'] + 1e-5)

# Q4 — pH categorization (already in dataset as Category_pH)
df['pH_Numeric'] = df['Category_pH'].map({'low_acidic': 0, 'neutral': 1, 'low_alkaline': 2})

# Q5 — Soil quality index
df['Soil_Quality_Index'] = (
    (df['Nitrogen']   / df['Nitrogen'].max()) * 0.3 +
    (df['Phosphorus'] / df['Phosphorus'].max()) * 0.3 +
    (df['Potassium']  / df['Potassium'].max()) * 0.3 +
    (1 - abs(df['pH'] - 7) / 3) * 0.1
)

# Q6 — Nutrient deficiency scores
df['N_Deficiency'] = (df['Nitrogen']   < df['Nitrogen'].quantile(0.25)).astype(int)
df['P_Deficiency'] = (df['Phosphorus'] < df['Phosphorus'].quantile(0.25)).astype(int)
df['K_Deficiency'] = (df['Potassium']  < df['Potassium'].quantile(0.25)).astype(int)
df['Total_Deficiency'] = df['N_Deficiency'] + df['P_Deficiency'] + df['K_Deficiency']

# Q8 — Season-based feature
df['Log_Rainfall'] = np.log1p(df['Rainfall'])

print('✅ Feature Engineering Complete!')
engineered = ['Total_NPK','NPK_Mean','N_P_Ratio','N_K_Ratio','P_K_Ratio',
              'Soil_Quality_Index','N_Deficiency','P_Deficiency','K_Deficiency',
              'Total_Deficiency','Log_Rainfall','pH_Numeric']
for f in engineered:
    print(f'  ✔ {f}')

---
## 🤖 Section 7 — Machine Learning Questions (Q1–Q10)

In [ ]:
# Feature and target setup
feature_cols = [
    'Nitrogen','Phosphorus','Potassium','pH','Temperature','Rainfall',
    'Light_Hours','Light_Intensity','Rh','Yield',
    'N_Ratio','P_Ratio','K_Ratio',
    'Soil_Enc','Season_Enc','pH_Numeric','Photo_Enc','Name_Enc',
    'Total_NPK','NPK_Mean','N_P_Ratio','N_K_Ratio','P_K_Ratio',
    'Soil_Quality_Index','N_Deficiency','P_Deficiency','K_Deficiency',
    'Total_Deficiency','Log_Rainfall'
]

X = df[feature_cols]
y = df['Fertility_Enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

print(f'✅ Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}  |  Features: {len(feature_cols)}')

In [ ]:
def evaluate_clf(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    preds = model.predict(Xte)
    acc   = accuracy_score(yte, preds)
    prec  = precision_score(yte, preds, average='weighted')
    rec   = recall_score(yte, preds, average='weighted')
    f1    = f1_score(yte, preds, average='weighted')
    print(f'  {name}')
    print(f'    Acc={acc*100:.2f}%  Prec={prec*100:.2f}%  Rec={rec*100:.2f}%  F1={f1*100:.2f}%')
    return {'Model': name, 'Accuracy': round(acc*100,2), 'Precision': round(prec*100,2),
            'Recall': round(rec*100,2), 'F1': round(f1*100,2), 'preds': preds}

print('📊 Model Training & Evaluation:\n')

# Q2 — Logistic Regression
lr_res  = evaluate_clf('Logistic Regression', LogisticRegression(max_iter=500, random_state=42), X_tr_sc, X_te_sc, y_train, y_test)
# Q3 — Decision Tree
dt_res  = evaluate_clf('Decision Tree',       DecisionTreeClassifier(max_depth=10, random_state=42), X_train, X_test, y_train, y_test)
# Q4 — Random Forest
rf_res  = evaluate_clf('Random Forest',       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), X_train, X_test, y_train, y_test)
# XGBoost
xgb_res = evaluate_clf('XGBoost',             XGBClassifier(n_estimators=100, random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss'), X_train, X_test, y_train, y_test)
# SVM
svm_res = evaluate_clf('SVM',                 SVC(kernel='rbf', random_state=42, probability=True), X_tr_sc, X_te_sc, y_train, y_test)

In [ ]:
# Q5 — Feature Importance (Random Forest)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
feat_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': rf_model.feature_importances_})
feat_imp = feat_imp.sort_values('Importance', ascending=True)

plt.figure(figsize=(11, 9))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feat_imp)))
plt.barh(feat_imp['Feature'], feat_imp['Importance'], color=colors, edgecolor='black')
plt.title('Q5 — Feature Importance (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# Q8 — Model comparison
all_res = [lr_res, dt_res, rf_res, xgb_res, svm_res]
comp_df = pd.DataFrame([{k:v for k,v in r.items() if k!='preds'} for r in all_res])
comp_df = comp_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)
print('🏆 Q8 — Model Comparison:')
print(comp_df.to_string(index=False))

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
colors = ['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4','#FFEAA7']
for ax, metric in zip(axes, ['Accuracy','Precision','Recall','F1']):
    ax.bar(comp_df['Model'], comp_df[metric], color=colors, edgecolor='black', width=0.55)
    for i, v in enumerate(comp_df[metric]):
        ax.text(i, v+0.2, f'{v}', ha='center', fontsize=8, fontweight='bold')
    ax.set_title(metric, fontweight='bold')
    ax.set_ylim([min(comp_df[metric])-5, 105])
    ax.tick_params(axis='x', rotation=20)
plt.suptitle('📊 Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix — Best Model
best_preds = xgb_res['preds']
cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_fertility.classes_, yticklabels=le_fertility.classes_)
plt.title('Confusion Matrix — XGBoost', fontsize=13, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()
print(classification_report(y_test, best_preds, target_names=le_fertility.classes_))

In [ ]:
# Cross Validation
print('📊 5-Fold Cross Validation:')
xgb_final = XGBClassifier(n_estimators=100, random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss')
cv_scores = cross_val_score(xgb_final, X, y, cv=5, scoring='accuracy')
print(f'  XGBoost CV: Mean={cv_scores.mean()*100:.2f}%  Std={cv_scores.std()*100:.2f}%')

---
## 🧠 Section 8 — Deep Learning Questions (Q1–Q8)

In [ ]:
# Q1–Q6 — ANN
ann_res = evaluate_clf('ANN (64-32)',
    MLPClassifier(hidden_layer_sizes=(64,32), activation='relu', solver='adam',
                  max_iter=300, random_state=42, early_stopping=True),
    X_tr_sc, X_te_sc, y_train, y_test)

# Q3 & Q4 — Deep Neural Network
dnn_model = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64, 32),
    activation='relu', solver='adam',
    max_iter=500, random_state=42,
    early_stopping=True, learning_rate_init=0.001
)
dnn_res = evaluate_clf('DNN (256-128-64-32)', dnn_model, X_tr_sc, X_te_sc, y_train, y_test)

In [ ]:
# Q5 — Loss curve
plt.figure(figsize=(10, 4))
plt.plot(dnn_model.loss_curve_, label='Training Loss', color='steelblue')
plt.title('Q5 — DNN Training Loss Curve', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

---
## 🎯 Section 9 — Soil Analysis & Recommendation System

In [ ]:
# Train final XGBoost
xgb_final = XGBClassifier(n_estimators=100, random_state=42, verbosity=0,
                           use_label_encoder=False, eval_metric='logloss')
xgb_final.fit(X_train, y_train)

FERTILIZER_RECS = {
    'N': 'Apply Urea (46-0-0) or Ammonium Nitrate to boost Nitrogen.',
    'P': 'Apply Single Super Phosphate (SSP) or DAP to boost Phosphorus.',
    'K': 'Apply Muriate of Potash (MOP) or Potassium Sulfate to boost Potassium.',
}

CROP_RECS = {
    'High':     ['Rice','Wheat','Sugarcane','Maize','Banana','Potato'],
    'Moderate': ['Millet','Sorghum','Groundnut','Sunflower','Lentil','Chickpea'],
}

def analyze_soil(nitrogen, phosphorus, potassium, ph, temperature, rainfall,
                 soil_type='Loam', season='Summer', crop_name='Wheat',
                 light_hours=12, light_intensity=500, rh=60, yield_val=2.0):
    """
    Analyze soil conditions and provide fertility classification,
    deficiency alerts, fertilizer & crop recommendations.
    """
    try:
        soil_enc  = le_soil.transform([soil_type.strip()])[0]
    except:
        soil_enc  = 0
    try:
        season_enc = le_season.transform([season.strip()])[0]
    except:
        season_enc = 0
    try:
        name_enc   = le_name.transform([crop_name.strip()])[0]
    except:
        name_enc   = 0

    ph_numeric = 0 if ph < 6.5 else (1 if ph <= 7.5 else 2)
    photo_enc  = 0

    total_npk  = nitrogen + phosphorus + potassium
    npk_mean   = total_npk / 3
    n_p_ratio  = nitrogen  / (phosphorus + 1e-5)
    n_k_ratio  = nitrogen  / (potassium  + 1e-5)
    p_k_ratio  = phosphorus/ (potassium  + 1e-5)
    sq_index   = (nitrogen/df['Nitrogen'].max()*0.3 +
                  phosphorus/df['Phosphorus'].max()*0.3 +
                  potassium/df['Potassium'].max()*0.3 +
                  (1 - abs(ph-7)/3)*0.1)
    n_def = int(nitrogen   < df['Nitrogen'].quantile(0.25))
    p_def = int(phosphorus < df['Phosphorus'].quantile(0.25))
    k_def = int(potassium  < df['Potassium'].quantile(0.25))
    n_ratio = df['N_Ratio'].mean()
    p_ratio = df['P_Ratio'].mean()
    k_ratio = df['K_Ratio'].mean()

    inp = np.array([[
        nitrogen, phosphorus, potassium, ph, temperature, rainfall,
        light_hours, light_intensity, rh, yield_val,
        n_ratio, p_ratio, k_ratio,
        soil_enc, season_enc, ph_numeric, photo_enc, name_enc,
        total_npk, npk_mean, n_p_ratio, n_k_ratio, p_k_ratio,
        sq_index, n_def, p_def, k_def,
        n_def+p_def+k_def, np.log1p(rainfall)
    ]])

    pred_enc  = xgb_final.predict(inp)[0]
    pred_prob = xgb_final.predict_proba(inp)[0]
    fertility = le_fertility.classes_[pred_enc]
    confidence= round(pred_prob[pred_enc]*100, 2)

    # Deficiency alerts
    deficiencies = []
    if n_def: deficiencies.append('N')
    if p_def: deficiencies.append('P')
    if k_def: deficiencies.append('K')

    print('=' * 60)
    print('      🌱 SOIL ANALYSIS RESULT')
    print('=' * 60)
    print(f'  Soil Type     : {soil_type}')
    print(f'  Season        : {season}')
    print(f'  N={nitrogen}  P={phosphorus}  K={potassium}  pH={ph}')
    print(f'  Temp={temperature}°C  Rainfall={rainfall}mm')
    print('-' * 60)
    print(f'  ✅ Predicted Fertility : {fertility.upper()}  ({confidence}% confidence)')
    print(f'  📊 Soil Quality Index  : {sq_index:.3f}')
    print(f'  🧪 Total NPK           : {total_npk:.1f}')
    print()
    if deficiencies:
        print(f'  ⚠️  Deficient Nutrients: {deficiencies}')
        print('  💊 Fertilizer Recommendations:')
        for d in deficiencies:
            print(f'     → {FERTILIZER_RECS[d]}')
    else:
        print('  ✅ No major nutrient deficiencies detected.')
    print()
    print(f'  🌾 Suitable Crops for {fertility} fertility:')
    for c in CROP_RECS[fertility]:
        print(f'     → {c}')
    print('=' * 60)
    return {'fertility': fertility, 'confidence': confidence, 'deficiencies': deficiencies}

In [ ]:
# Test Case 1 — Loam soil, good nutrients
r1 = analyze_soil(nitrogen=80, phosphorus=60, potassium=70, ph=6.8,
                   temperature=22, rainfall=900, soil_type='Loam', season='Summer')

In [ ]:
# Test Case 2 — Sandy soil, nutrient deficient
r2 = analyze_soil(nitrogen=20, phosphorus=15, potassium=18, ph=5.5,
                   temperature=28, rainfall=500, soil_type='Sandy', season='Fall')

In [ ]:
# Test Case 3 — Custom input (change values as needed)
r3 = analyze_soil(
    nitrogen=55, phosphorus=45, potassium=60, ph=7.0,
    temperature=25, rainfall=750, soil_type='Sandy Loam',
    season='Spring', crop_name='Wheat'
)

---
## 💾 Section 10 — Save Models & Final Summary

In [ ]:
joblib.dump(xgb_final,    'soil_xgb_model.pkl')
joblib.dump(rf_model,     'soil_rf_model.pkl')
joblib.dump(scaler,       'soil_scaler.pkl')
joblib.dump(le_fertility, 'soil_le_fertility.pkl')
joblib.dump(le_soil,      'soil_le_soil.pkl')
joblib.dump(le_season,    'soil_le_season.pkl')

print('💾 Models saved:')
for f in ['soil_xgb_model.pkl','soil_rf_model.pkl','soil_scaler.pkl',
          'soil_le_fertility.pkl','soil_le_soil.pkl','soil_le_season.pkl']:
    print(f'  ✅ {f}')

In [ ]:
all_results = [lr_res, dt_res, rf_res, xgb_res, svm_res, ann_res, dnn_res]
final_df = pd.DataFrame([{k:v for k,v in r.items() if k!='preds'} for r in all_results])
final_df = final_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)

print('=' * 65)
print('   ✅ MODULE 3: SOIL ANALYSIS SYSTEM — FINAL SUMMARY')
print('=' * 65)
print(f'  Dataset     : Soil_Nutrients.csv')
print(f'  Records     : {df.shape[0]:,}  |  Features: {df.shape[1]}')
print(f'  Crops       : {df["Name"].nunique()}  |  Soil Types: {df["Soil_Type"].nunique()}')
print(f'  Target      : Fertility (High / Moderate)')
print()
print('  Model Accuracy Results:')
for _, row in final_df.iterrows():
    star = ' ⭐ Best' if _ == 0 else ''
    print(f'    {row["Model"]:30s}: Acc={row["Accuracy"]}%  F1={row["F1"]}%{star}')
print()
print('  Sections Covered:')
sections = [
    ('Data Collection',10),('Preprocessing',12),('EDA',17),
    ('Visualization',10),('Feature Engineering',8),('Machine Learning',10),
    ('Soil Recommendations',8),('Deep Learning',8),('Model Training',6),
    ('Model Evaluation',8),('Soil Analysis System',8),('Integration',7),('Future Work',10)
]
total = 0
for name, count in sections:
    print(f'    ✔ {name:25s}: {count} questions')
    total += count
print(f'\n  Total Questions Answered: {total}')
print('=' * 65)